In [ ]:
import re
import os
import glob
import json
import copy
import random
import argparse
import itertools

from typing import List, Dict
from dataclasses import dataclass
import xml.etree.ElementTree as ET
from collections import defaultdict
from tree_sitter_languages import get_parser
from tree_climber.ast_creator import AstErrorException

from tree_climber.tree_sitter_utils import c_parser, cpp_parser
from tree_climber.ast_creator import ASTCreator
from tree_climber.cfg_creator import CFGCreator


In [2]:
language="c"
parser = get_parser(language)

In [3]:
def read_file(path):
    with open(path, "r") as f:
        data = f.read()
    return data
def read_jsonl(path):
    with open(path, "r") as f:
        data = [json.loads(ex) for ex in f]
    return data

In [4]:
def get_parsed_info(src, lang):
    if lang == "c":
        parser = c_parser
    elif lang == "cpp":
        parser = cpp_parser
    else:
        raise NotImplementedError(lang)
        
    tree = parser.parse(bytes(src, "utf-8"))
    ast = ASTCreator.make_ast(tree.root_node)
    cfg = CFGCreator.make_cfg(ast)
    
        # map nodes based on source code lines to annotated linenos "// LXXX"
    def get_lineno(line):
        """Parse line of code to get lineno comment (// LXXX)"""
        m = re.search(r"// (L[0-9]+)", line)
        if m:
            return m.group(1)
        return None

    src_lines = src.splitlines()

    def get_node_lineno(n):
        """Helper to get LXXX for a CFG node"""
        attr = cfg.nodes[n]
        # print(n)
        # print(attr)
        if "n" not in attr:
            return None  # no ast node - most often this means it's FUNC_EXIT or some other CFG node
        # print(attr["n"])
        # print(attr["n"].start_point[0])
        # print(get_lineno(src_lines[attr["n"].start_point[0]]))
        return get_lineno(src_lines[attr["n"].start_point[0]])

    # get destination nodes
    def get_dest_nodes(n):
        """Return dict of edge_attr -> destination node for a branch node n"""
        d = {}
        for m, m_attr in cfg.adj[n].items():
            label = ",".join(sorted([v["label"] for v in m_attr.values()]))
            d[label] = m
        return d
    # get branch nodes
    branch_nodes = []
    for n in cfg.nodes():
        if len(cfg.adj[n]) > 1:
            branch_nodes.append(n)
    
    # # print("Branch node", branch_nodes)
    # print("Branch Node Line No ", [get_node_lineno(n) for n in branch_nodes])
    dest_nodes = [get_dest_nodes(n) for n in branch_nodes]
    # print(dest_nodes)
    
    # map to lineno
    branch_nodes_linenos = [get_node_lineno(n) for n in branch_nodes]
    dest_node_linenos = [
        {k: get_node_lineno(v) for k, v in d.items()} for d in dest_nodes
    ]
    # print("Destination nodes line number", dest_node_linenos_)
    return branch_nodes_linenos, dest_node_linenos


In [53]:
full_data = read_jsonl("../results/sequences/sequences_langC_from0_to4052_limitNone_full.jsonl")

In [79]:
def extract_identifiers(node, results):
    if node.type == "identifier":
        results.add(node.text.decode())
    for child in node.children:
        extract_identifiers(child, results)

In [136]:
def bfs(code_str):
    lines = code_str.split("\n")
    if isinstance(code_str, str):
        code_str = code_str.encode()
    root = parser.parse(code_str).root_node
    
    q = [root]
    concepts = {}
    while len(q) > 0:
        n = q.pop()
        
        if n.type in ("for_statement", "while_statement", "if_statement", "do_statement"):
            condition = n.child_by_field_name("condition")
            if condition:
                line_no = condition.start_point[0] + 1
                concepts[line_no] = condition.text.decode()
                identifiers = set()
                extract_identifiers(condition, identifiers)
                concepts[line_no] = {
                    "condition_type": n.type,
                    "code": condition.text.decode(),
                    "variables": identifiers,
                }
                # print(condition.text.decode()
        q.extend(reversed(list(n.children)))
    return concepts

In [137]:
def read_xml(xml_file, schema='tree', considered_lines =[], concepts = None, adj=None):
    with open(xml_file) as f:
        tree = ET.parse(f)
    root = tree.getroot()
    root.attrib["schema"] = schema
    states = {}
    states_after_line = {}
    before_states = {}
    next_sibling = {}
    for i, child in enumerate(root):
        if i > 0:
            next_sibling[i-1] = child
        next_sibling[i] = None
    for i, child in enumerate(root):
        # print(child)
        assert child.tag == "line", f"'{child.tag}' should be 'line'"
        # print(child.attrib)
        
        lineno = int(child.attrib["line"])
        # print(lineno)
        # new_or_modified = False
        before_states = copy.deepcopy(states)
        for variable in child:
            # print(variable.attrib)
            # print(dir(variable))
            # if variable.attrib["age"] in ("new", "modified"):
            #     new_or_modified = True
                # print(variable.attrib["name"], variable.attrib["age"], variable.text)
            states[variable.attrib["name"]] = variable.text
        
        # if lineno in considered_lines and new_or_modified:
        #     states_after_line[lineno] = [before_states, copy.deepcopy(states)]
        if lineno in considered_lines:
            sib = next_sibling[i]
            if sib != None:
                # print(lineno, int(sib.attrib["line"]))
                next_line =  int(sib.attrib["line"])
                # print(lineno, adj)
                adja_ = adj[lineno]
                output = None
                if adja_['True'] == next_line:
                    output = True
                elif adja_['False'] == next_line:
                    output = False
                
                if "states" not in concepts[lineno]:
                    concepts[lineno]['states'] = []
                concepts[lineno]['states'].append({"output": output})
                variables = states
                state_vars = set(variables.keys())
                concept_vars = set(concepts[lineno]['variables'])
                # print(variables.keys())
                # print(concepts[lineno]['variables'])
                considered_vars = state_vars & concept_vars
                concepts[lineno]['states'][-1]['variables_state'] = {}
                for var in considered_vars:
                    if var in before_states:
                        concepts[lineno]['states'][-1]['variables_state'][var] = states[var]
                # assert len(state_vars & concept_vars) == len(concept_vars)
                # print(len(considered_vars))
                
                # if output == None:
                #     print(concepts[lineno], output)
        for lineno in concepts:
            if "states" in concepts[lineno]:
                concepts[lineno]['states'] = random.sample(concepts[lineno]['states'], min(len(concepts[lineno]['states']), 2))
    return states_after_line
        

In [ ]:
cnt = 0
er = 0
call_count, ass_count = 0, 0
condition = {}
for ex in full_data:
    # print(ex)
    try:
        src = read_file(f"../Project_CodeNet/data/{ex['filepath']}")

        cnt += 1
        concepts = bfs(src)
        # print(concepts)
        # print(ex["src"])
        branch, dest = get_parsed_info(ex['src'], ex['lang'])

        adj = {}
        for bra, dst in zip(branch, dest):
            if len(dst) != 2 or  "True" not in dst or "False" not in dst:
                dst
                continue
            if bra == dst['True'] or bra == dst['False']:
                continue

            # # print(bra, dst)
            # print(bra)
            if bra == None:
                continue
            bra_no = int(bra[1:])
            # print()
            tr_no = int(dst['True'][1:]) if dst['True'] != None else None
            fal_no = int(dst['False'][1:]) if dst['False'] != None else None
            adj[bra_no] = {
                "True": tr_no,
                "False": fal_no
            }

        # print(concepts.keys())
        concept_lines = set(concepts.keys())
        # print(concept_lines)
        branch_lines = set([x for x in adj])
        # print(branch_lines)

        common_lines = concept_lines & branch_lines
        # print(common_lines)
        # print(adj)
        path_ = ex['filepath'].split(".")[0]
        read_xml(f"../results/tree/{path_}/{ex['input_no']}.txt_log.xml", considered_lines=common_lines, concepts=concepts, adj=adj)
        for con in concepts:
            # print(con)
            concepts[con]['variables']= list(concepts[con]['variables'])

        condition[ex["idx"]] = concepts
    except  AstErrorException as e:
        er += 1
    except Exception as e:
        er += 1
        print(str(e))
    # print(ex['idx'], ex['filepath'], ex['input_no'])
    # print(concepts)
    if cnt == 1000:
        break
    

In [ ]:
er

In [ ]:
condition

In [141]:
# for ex in final_data:
#     for con in ex['condition']:
#         # print(con)
#         ex['condition'][con]['variables']= list(ex['condition'][con]['variables'])
#         # for key in ex['concepts'][con]:
#         #     print(key, type(ex['concepts'][con][key]), ex['concepts'][con][key])
#     # break

In [142]:
def get_value_type(typ):
    if "char" in typ:
        return "char"
    elif "double" in typ or "float" in typ:
        return "float"
    elif "int" in typ or "long" in typ:
        return "int"
    elif "bool" in typ:
        return "bool"
    elif "void" in typ:
        return "void"
    return "unknown"

def get_data_type(typ):
    if "*" in typ:
        return "pointer"
    elif "[" in typ:
        return "array"
    else:
        return "basic"
    
def get_quantized_value_for_pointer(variable):
    if variable.text == "0x0":
        return "NULL"
    else:
        return "NOT_NULL"

def get_quantized_value_for_array(variable, var_states):
    if variable.attrib["age"] == "old":
        return var_states[variable.attrib["name"]][2]
    elif variable.attrib["age"] == "modified":
        return "INITIALIZED"
    else:
        return "NOT_INITIALIZED"
        
def get_quantized_value_for_basic(variable, data_type, value_type):
    try:
        if value_type == "int":
            value = int(variable.text)
            if 10000 < value:
                return "POSITIVE-VL"
            elif 0 < value <= 10000:
                return "POSITIVE-REG"
            if value == 0:
                return "ZERO"
            elif -10000 <= value < 0:
                return "NEGATIVE-REG"
            elif value < -10000:
                return "NEGATIVE-VL"
        elif value_type == float:
            value = float(variable.text)
            if 10000 < value:
                return "POSITIVE-VL"
            elif 1 < value <= 10000:
                return "POSITIVE-REG"
            elif 0 < value <= 1:
                return "POSITIVE-VS"
            elif value == 0:
                return "ZERO"
            elif 0 > value >= -1:
                return "NEGATIVE-VS"
            elif -10000 <= value < -1:
                return "NEGATIVE-REG"
            elif value < -10000:
                return "NEGATIVE-VL"
        elif value_type == "char":
            value = variable.text[1:-1]
            value  = "".join(value.split())
            # print(value, end=" ")
            # print(value, value.isalpha(), type(value), len(value))
            if value == "\\000":
                # print("NULL")
                return "NULL"
            elif value.isalpha():
                # print("Alpha")
                return "ALPHA"
            else:
                # print("NOT Alpha")
                # print(value)
                return "NOT_ALPHA"
        elif value_type == "bool":
            value = int(variable.text)
            if value == 1:
                return "TRUE"
            else:
                return "FALSE"
        elif value_type == "void":
            return "VOID"
        else:
            return "UNKNOWN"
    except:
        return "UNKNOWN"
        
    
def quantized_variable(variable, var_states):
    typ = variable.attrib["typ"]
    data_type, value_type = get_data_type(typ), get_value_type(typ)
    if data_type == "array":
        quantized_value = get_quantized_value_for_array(variable, var_states)
    elif data_type == "pointer":
        quantized_value = get_quantized_value_for_pointer(variable)
        
    else:
        quantized_value = get_quantized_value_for_basic(variable, data_type, value_type)
    return data_type, value_type, quantized_value

In [143]:
def extract_base_identifier(node):
    """
    Recursively walk the left-hand side and return the base variable name.
    """
    if node.type == 'identifier':
        return [node]
    vars = []
    for child in node.children:
        result = extract_base_identifier(child)
        if result:
            vars.extend(result)

    return vars
def extract_left_identifier(node):
    """
    Recursively walk the left-hand side and return the base variable name.
    """
    if node.type == 'identifier':
        return node.text.decode()

    for child in node.children:
        result = extract_left_identifier(child)
        if result:
            return result

    return None
def extract_identifiers(node, results):
    if node.type == "identifier":
        results.add(node.text.decode())
    for child in node.children:
        extract_identifiers(child, results)
def is_call(n):
    if "call" in n.type:
        return True
    for child in n.children:
        # print(child.type,)
        if is_call(child):
            return True
    return False


In [144]:
def bfs(code_str):
    src = code_str
    lines = code_str.split("\n")
    if isinstance(code_str, str):
        code_str = code_str.encode()
    # print(code_str)
    root = parser.parse(code_str).root_node
    
    q = [root]
    concepts = {}
    while len(q) > 0:
        n = q.pop()
        # print(dir(n))
        # if any("call" in ch.type for ch in n.children):
        #         print(n.text.decode())
        if "expression_statement" in n.type and any("assignment" in ch.type for ch in n.children):
            # print(n.type, n.text.decode())
            # if is_call(n):
            #     print("found ")
            line_no = n.start_point[0] + 1
            identifiers = set()
            extract_identifiers(n, identifiers)
            left_var = extract_left_identifier(n)
            # print(left_var)

            concepts[line_no] = {
                "code": n.text.decode(),
                "variables": list(identifiers),
                "modified_variable": left_var,
            }
        q.extend(reversed(list(n.children)))
    # print(concepts)
    return concepts

In [145]:
def map_original_to_striped(x, y):
    removed_char = ["\n", " ", "\t", "\r", "\v", "\f"]
    
    # print(x, y)
    ptr, ptr_1 = 0, 0
    map_ = {}
    while ptr < len(x) and ptr_1 < len(y):
        if x[ptr] in removed_char:
            ptr += 1
            continue
        if y[ptr_1] in removed_char:
            ptr_1 += 1
            continue
        
        if x[ptr] == y[ptr_1]:
            map_[ptr] = ptr_1
            ptr += 1
            ptr_1 += 1
        else:
            return None
    return map_
    

In [146]:
def read_xml(xml_file, schema='tree', considered_lines =None, concepts=None):
    with open(xml_file) as f:
        tree = ET.parse(f)
    root = tree.getroot()
    root.attrib["schema"] = schema
    
    data_types = set()
    line_to_value = {}
    var_states = {}
    states, before_states = {}, {}
    for child in root:
        # print(child)
        assert child.tag == "line", f"'{child.tag}' should be 'line'"
        # print(child.attrib)
        lineno = int(child.attrib["line"])
        # print(lineno)
        before_states = copy.deepcopy(states)

        # changed_variables = 
        for variable in child:
            states[variable.attrib["name"]] = variable.text
            data_type, value_type, quantized_value =  quantized_variable(variable, var_states)
            var_name = variable.attrib['name']
            var_states[var_name] = [data_type, value_type, quantized_value]
            
        # print(lineno, var_states)
        # print(lineno, considered_lines, lineno in considered_lines)
        # if lineno in considered_lines and len(states):
        if lineno in considered_lines:
            if "states" not in concepts[lineno]:
                concepts[lineno]['states'] = []
            # print(concepts[lineno])
            # if concepts[lineno]['modified_variable'] not in var_states:
            #     print(child)
            #     for variable in child:
            #         print(variable)
            #     print(var_states)
            #     print(concepts[lineno]['modified_variable'])
            #     print(states)
            # print(lineno,states, concepts[lineno])
            concepts[lineno]['states'].append({"output": var_states[concepts[lineno]['modified_variable']]})
            variables = states
            state_vars = set(variables.keys())
            concept_vars = set(concepts[lineno]['variables'])
            # print(variables.keys())
            # print(concepts[lineno]['variables'])
            considered_vars = state_vars & concept_vars
            concepts[lineno]['states'][-1]['variables_state'] = {}
            for var in considered_vars:
                if var in before_states:
                    concepts[lineno]['states'][-1]['variables_state'][var] = states[var]
    for lineno in concepts:
        if "states" in concepts[lineno]:
            concepts[lineno]['states'] = random.sample(concepts[lineno]['states'], min(len(concepts[lineno]['states']), 2))
    return concepts

In [ ]:
var_types = set()
cnt = 0
assignment = {}
cnt_ = 0
states_found = 0
err_ = 0
for ex in full_data:
    # if ex['idx'] != 705 :
    #     continue
    cnt += 1
    # if cnt_ <= 13000:
    #     continue
    try:
            # print(ex['idx'])
        src = read_file(f"../Project_CodeNet/data/{ex['filepath']}")
        # print(src)
        lines = src.splitlines(keepends=True)
        concepts = bfs(src)
        # print(concepts)


        path_ = ex['filepath'].split(".")[0]
        concept_lines = set(concepts.keys())
        # print(concept_lines)
        # print(path_)
        states = read_xml(f"../results/tree/{path_}/{ex['input_no']}.txt_log.xml", considered_lines=concept_lines, concepts=concepts)
        # print("After States: ", states)

        # print(states)
        # print(states)
        assignment[ex["idx"]] = states
        
        flag = 0
        for line in states:
            if "states" in states[line] and len(states[line]['states']) > 0:
                flag = 1
        states_found += flag
        # cnt += 1
        if cnt % 1000 == 0:
            print("Done with ", cnt, "states found", states_found)
        if cnt == 1000:
            break
    except Exception as E:
        # print(str(E), ex['idx'])
        # pass
        # print(ex['idx'], str(E))
        err_ += 1


In [148]:
branch_data = read_jsonl("../results/sequences/sequences_BRANCH.jsonl")

In [ ]:
branch_data[4].keys()

In [150]:
labels_map = {
    False: 0,
    True: 1,
    'ALPHA': 2,
     'INITIALIZED': 3,
     'NEGATIVE-REG': 4,
     'NEGATIVE-VL': 5,
     'NOT_ALPHA': 6,
     'NOT_INITIALIZED': 7,
     'NOT_NULL': 8,
     'NULL': 9,
     'POSITIVE-REG': 10,
     'POSITIVE-VL': 11,
     'UNKNOWN': 12,
     'ZERO': 13,
}

In [ ]:
len(condition), len(assignment)

In [ ]:
condition

In [ ]:
final_data = []
for ex in branch_data:
    try:
        # cond = condition.get(ex["idx"], {})
        # assin = assignment.get(ex["idx"], {})
        cond = condition[ex["idx"]]
        assign = assignment[ex["idx"]]
        refined_cond = {}
        for line in cond:
            lineno = int(line)
            if "states" not in cond[line]:
                continue
            for state in cond[line]['states']:
                if state['output'] != None:
                    state['output'] = labels_map[state['output']]
                    # print("here")
            refined_cond[lineno] = cond[line]
        refined_assign = {}
        for line in assign:
            lineno = int(line)
            if "states" not in assign[line]:
                continue
            for state in assign[line]['states']:
                # print(state["output"])
            #     print(state)
                if state['output'] != None:
                    state['output'] = labels_map[state['output'][-1]]
            refined_assign[lineno] = assign[line]
        # print(cond, refined_cond)
        ex1 = {
            'idx': ex['idx'],
            'src_lines': ex['src_lines'],
            'src_linenos': ex['src_linenos'],
            'covered_in_trace': ex['covered_in_trace'],
            'filepath': ex['filepath'],
            'input': ex['input'],
            'condition': refined_cond,
            'quantized': refined_assign,
        }
        final_data.append(ex1)
    except Exception as e:
        print(str(e))

In [ ]:
len(final_data)

In [157]:
with open("../results/sequences/test_1000.jsonl", "w") as f:
    f.write("\n".join([json.dumps(ex) for ex in final_data]))